In [0]:
# ===========================================
# Notebook Name:
# 00_build_tournaments
#
# Purpose:
# Parse tournament events from Bronze API
# responses and merge them into Silver.
#
# Input:
# pokemon.bronze.tournament_list_raw
#
# Output:
# pokemon.silver.tournaments
#
# Silver grain:
# One row per tournament_id
#
# Scope:
# event_date >= 2026-01-01
#
# Merge key:
# tournament_id
# (sourced from event_holding_id, the
# persistent tournament identifier used by
# the event-results API -- NOT event['id'],
# which is an ephemeral list-position id that
# cannot be used to fetch results)
#
# Change detection:
# event_hash
# ===========================================

In [0]:
from datetime import datetime, timezone
import hashlib
import json

from pyspark.sql import functions as F
from pyspark.sql.types import (
    DateType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)
from pyspark.sql.window import Window

In [0]:
TOURNAMENT_LIST_RAW_TABLE = (
    "pokemon.bronze.tournament_list_raw"
)

TOURNAMENTS_TABLE = (
    "pokemon.silver.tournaments"
)

MIN_EVENT_DATE = "2026-01-01"

print("Input :", TOURNAMENT_LIST_RAW_TABLE)
print("Output:", TOURNAMENTS_TABLE)
print("Scope :", MIN_EVENT_DATE)

In [0]:
bronze_df = (
    spark.table(
        TOURNAMENT_LIST_RAW_TABLE
    )
    .select(
        "run_id",
        "offset",
        "response_hash",
        "payload",
        "fetched_at",
        "ingestion_date",
    )
)

print(
    "Bronze page rows:",
    bronze_df.count(),
)

In [0]:
def canonical_json(
    value: dict,
) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

def calculate_event_hash(
    event: dict,
) -> str:
    event_json = canonical_json(
        event
    )

    return hashlib.sha256(
        event_json.encode("utf-8")
    ).hexdigest()

In [0]:
def get_first_value(
    record: dict,
    keys: list[str],
):
    for key in keys:
        value = record.get(key)

        if value not in (
            None,
            "",
        ):
            return value

    return None

def to_optional_string(
    value,
):
    if value is None:
        return None

    return str(value)

def to_optional_integer(
    value,
):
    if value in (
        None,
        "",
    ):
        return None

    try:
        return int(value)

    except (
        TypeError,
        ValueError,
    ):
        return None

In [0]:
def parse_event_date(
    value,
):
    if value in (
        None,
        "",
    ):
        return None

    text = str(value).strip()

    formats = [
        "%Y%m%d",
        "%Y-%m-%d",
        "%Y/%m/%d",
    ]

    for date_format in formats:
        try:
            return datetime.strptime(
                text,
                date_format,
            ).date()

        except ValueError:
            continue

    return None

In [0]:
tournament_rows = []
parse_errors = []

for bronze_row in (
    bronze_df.toLocalIterator()
):
    try:
        payload_object = json.loads(
            bronze_row["payload"]
        )

        events = payload_object.get(
            "event",
            [],
        )

        if events is None:
            events = []

        if not isinstance(events, list):
            raise ValueError(
                "payload['event'] is not a list"
            )

        for event in events:
            if not isinstance(
                event,
                dict,
            ):
                continue

            # event_holding_id is the persistent
            # tournament identifier accepted by the
            # event-results API. event['id'] is a
            # separate, ephemeral list-position id
            # and must not be used as tournament_id.
            tournament_id = (
                to_optional_integer(
                    event.get("event_holding_id")
                )
            )

            event_date = parse_event_date(
                get_first_value(
                    event,
                    [
                        "event_date_params",
                        "event_date_full",
                        "held_date",
                        "start_date",
                    ],
                )
            )

            if tournament_id is None:
                continue

            if event_date is None:
                continue

            if str(event_date) < MIN_EVENT_DATE:
                continue

            event_json = canonical_json(
                event
            )

            event_hash = (
                calculate_event_hash(
                    event
                )
            )

            tournament_name = (
                get_first_value(
                    event,
                    [
                        "event_name",
                        "name",
                        "title",
                        "event_title",
                    ],
                )
            )

            venue_name = (
                get_first_value(
                    event,
                    [
                        "shop_name",
                        "venue_name",
                        "place_name",
                        "venue",
                    ],
                )
            )

            prefecture = (
                get_first_value(
                    event,
                    [
                        "prefecture",
                        "prefecture_name",
                        "area_name",
                    ],
                )
            )

            tournament_rows.append({
                "tournament_id": (
                    tournament_id
                ),
                "event_date": (
                    event_date
                ),
                "event_date_text": (
                    to_optional_string(
                        event.get(
                            "event_date"
                        )
                    )
                ),
                "event_date_week": (
                    to_optional_string(
                        event.get(
                            "event_date_week"
                        )
                    )
                ),
                "event_started_at": (
                    to_optional_string(
                        event.get(
                            "event_started_at"
                        )
                    )
                ),
                "event_ended_at": (
                    to_optional_string(
                        event.get(
                            "event_ended_at"
                        )
                    )
                ),
                "tournament_name": (
                    to_optional_string(
                        tournament_name
                    )
                ),
                "venue_name": (
                    to_optional_string(
                        venue_name
                    )
                ),
                "prefecture": (
                    to_optional_string(
                        prefecture
                    )
                ),
                "date_id": (
                    to_optional_integer(
                        event.get("date_id")
                    )
                ),
                "shop_id": (
                    to_optional_integer(
                        event.get("shop_id")
                    )
                ),
                "event_type": (
                    to_optional_string(
                        get_first_value(
                            event,
                            [
                                "event_type",
                                "event_type_id",
                                "event_category",
                            ],
                        )
                    )
                ),
                "event_json": (
                    event_json
                ),
                "event_hash": (
                    event_hash
                ),
                "source_run_id": (
                    bronze_row["run_id"]
                ),
                "source_offset": (
                    bronze_row["offset"]
                ),
                "source_response_hash": (
                    bronze_row[
                        "response_hash"
                    ]
                ),
                "source_fetched_at": (
                    bronze_row[
                        "fetched_at"
                    ]
                ),
            })

    except Exception as error:
        parse_errors.append({
            "run_id": bronze_row[
                "run_id"
            ],
            "offset": bronze_row[
                "offset"
            ],
            "error": (
                f"{type(error).__name__}: "
                f"{str(error)}"
            )[:2000],
        })

print(
    "Parsed tournament rows:",
    len(tournament_rows),
)

print(
    "Parse errors:",
    len(parse_errors),
)

if parse_errors:
    display(
        spark.createDataFrame(
            parse_errors
        )
    )

    raise ValueError(
        f"{len(parse_errors)} "
        "Bronze payloads could not be parsed"
    )

In [0]:
tournament_schema = StructType([
    StructField(
        "tournament_id",
        IntegerType(),
        False,
    ),
    StructField(
        "event_date",
        DateType(),
        False,
    ),
    StructField(
        "event_date_text",
        StringType(),
        True,
    ),
    StructField(
        "event_date_week",
        StringType(),
        True,
    ),
    StructField(
        "event_started_at",
        StringType(),
        True,
    ),
    StructField(
        "event_ended_at",
        StringType(),
        True,
    ),
    StructField(
        "tournament_name",
        StringType(),
        True,
    ),
    StructField(
        "venue_name",
        StringType(),
        True,
    ),
    StructField(
        "prefecture",
        StringType(),
        True,
    ),
    StructField(
        "date_id",
        IntegerType(),
        True,
    ),
    StructField(
        "shop_id",
        IntegerType(),
        True,
    ),
    StructField(
        "event_type",
        StringType(),
        True,
    ),
    StructField(
        "event_json",
        StringType(),
        False,
    ),
    StructField(
        "event_hash",
        StringType(),
        False,
    ),
    StructField(
        "source_run_id",
        StringType(),
        False,
    ),
    StructField(
        "source_offset",
        IntegerType(),
        False,
    ),
    StructField(
        "source_response_hash",
        StringType(),
        False,
    ),
    StructField(
        "source_fetched_at",
        TimestampType(),
        False,
    ),
])

tournament_candidates_df = (
    spark.createDataFrame(
        tournament_rows,
        schema=tournament_schema,
    )
)

print(
    "Candidate rows:",
    tournament_candidates_df.count(),
)

print(
    "Candidate tournament IDs:",
    tournament_candidates_df
    .select("tournament_id")
    .distinct()
    .count(),
)

In [0]:
latest_tournament_window = (
    Window
    .partitionBy(
        "tournament_id"
    )
    .orderBy(
        F.col(
            "source_fetched_at"
        ).desc(),
        F.col(
            "source_offset"
        ).asc(),
    )
)

latest_tournaments_df = (
    tournament_candidates_df
    .withColumn(
        "latest_row_number",
        F.row_number().over(
            latest_tournament_window
        ),
    )
    .filter(
        F.col(
            "latest_row_number"
        ) == 1
    )
    .drop(
        "latest_row_number"
    )
)

display(
    latest_tournaments_df
    .orderBy(
        F.col("event_date").desc(),
        "tournament_id",
    )
)

In [0]:
duplicate_ids_df = (
    latest_tournaments_df
    .groupBy(
        "tournament_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

if duplicate_ids_df.count() > 0:
    display(
        duplicate_ids_df
    )

    raise ValueError(
        "Duplicate tournament IDs detected"
    )

print(
    "Validation passed: "
    "one row per tournament_id"
)

invalid_dates_df = (
    latest_tournaments_df
    .filter(
        F.col("event_date")
        < F.lit(
            MIN_EVENT_DATE
        ).cast("date")
    )
)

if invalid_dates_df.count() > 0:
    display(
        invalid_dates_df
    )

    raise ValueError(
        "Events before 2026-01-01 detected"
    )

print(
    "Validation passed: "
    "all events are from 2026 onward"
)

invalid_required_fields_df = (
    latest_tournaments_df
    .filter(
        F.col(
            "tournament_id"
        ).isNull()
        | F.col(
            "event_date"
        ).isNull()
        | F.col(
            "event_hash"
        ).isNull()
    )
)

if (
    invalid_required_fields_df.count()
    > 0
):
    display(
        invalid_required_fields_df
    )

    raise ValueError(
        "Required tournament fields "
        "are missing"
    )

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{TOURNAMENTS_TABLE}
(
    tournament_id INT NOT NULL,
    event_date DATE NOT NULL,
    tournament_name STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
COMMENT 'Official Pokemon Card tournaments'
""")

required_columns = {
    "tournament_name": "STRING",
    "created_at": "TIMESTAMP",
    "event_date": "DATE",
    "event_date_text": "STRING",
    "event_date_week": "STRING",
    "event_started_at": "STRING",
    "event_ended_at": "STRING",
    "venue_name": "STRING",
    "prefecture": "STRING",
    "date_id": "INT",
    "event_type": "STRING",
    "event_json": "STRING",
    "event_hash": "STRING",
    "source_run_id": "STRING",
    "source_offset": "INT",
    "source_response_hash": "STRING",
    "source_fetched_at": "TIMESTAMP",
    "first_seen_at": "TIMESTAMP",
    "last_changed_at": "TIMESTAMP",
}

existing_columns = {
    field.name
    for field in (
        spark.table(
            TOURNAMENTS_TABLE
        ).schema.fields
    )
}

for (
    column_name,
    column_type,
) in required_columns.items():
    if (
        column_name
        not in existing_columns
    ):
        spark.sql(f"""
        ALTER TABLE
        {TOURNAMENTS_TABLE}
        ADD COLUMNS (
            {column_name} {column_type}
        )
        """)

        print(
            "Added column:",
            column_name,
            column_type,
        )

In [0]:
if spark.catalog.tableExists(
    TOURNAMENTS_TABLE
):
    existing_tournaments_df = (
        spark.table(
            TOURNAMENTS_TABLE
        )
        .select(
            "tournament_id",
            "event_hash",
        )
    )

else:
    existing_tournaments_df = (
        spark.createDataFrame(
            [],
            """
            tournament_id INT,
            event_hash STRING
            """,
        )
    )

change_analysis_df = (
    latest_tournaments_df.alias("source")
    .join(
        existing_tournaments_df.alias(
            "target"
        ),
        on="tournament_id",
        how="left",
    )
    .withColumn(
        "change_type",
        F.when(
            F.col(
                "target.tournament_id"
            ).isNull(),
            F.lit("insert"),
        )
        .when(
            F.col(
                "source.event_hash"
            )
            != F.col(
                "target.event_hash"
            ),
            F.lit("update"),
        )
        .otherwise(
            F.lit("unchanged")
        ),
    )
)

display(
    change_analysis_df
    .groupBy("change_type")
    .count()
)

change_counts = {
    row["change_type"]: row["count"]
    for row in (
        change_analysis_df
        .groupBy("change_type")
        .count()
        .collect()
    )
}

insert_count = change_counts.get(
    "insert",
    0,
)

update_count = change_counts.get(
    "update",
    0,
)

unchanged_count = change_counts.get(
    "unchanged",
    0,
)

print("Insert   :", insert_count)
print("Update   :", update_count)
print("Unchanged:", unchanged_count)

In [0]:
latest_tournaments_df.createOrReplaceTempView(
    "tournament_merge_source"
)

In [0]:
spark.sql(f"""
MERGE INTO
{TOURNAMENTS_TABLE} AS target

USING
tournament_merge_source AS source

ON
target.tournament_id = source.tournament_id

WHEN MATCHED
AND (
    target.event_hash IS NULL
    OR target.event_hash <> source.event_hash
)
THEN UPDATE SET

    target.event_date =
        source.event_date,

    target.event_date_text =
        source.event_date_text,

    target.event_date_week =
        source.event_date_week,

    target.event_started_at =
        source.event_started_at,

    target.event_ended_at =
        source.event_ended_at,

    target.tournament_name =
        source.tournament_name,

    target.venue_name =
        source.venue_name,

    target.prefecture =
        source.prefecture,

    target.date_id =
        source.date_id,

    target.shop_id =
        source.shop_id,

    target.event_type =
        source.event_type,

    target.event_json =
        source.event_json,

    target.event_hash =
        source.event_hash,

    target.source_run_id =
        source.source_run_id,

    target.source_offset =
        source.source_offset,

    target.source_response_hash =
        source.source_response_hash,

    target.source_fetched_at =
        source.source_fetched_at,

    target.last_changed_at =
        source.source_fetched_at,

    target.updated_at =
        current_timestamp()

WHEN NOT MATCHED
THEN INSERT (
    tournament_id,
    event_date,
    event_date_text,
    event_date_week,
    event_started_at,
    event_ended_at,
    tournament_name,
    venue_name,
    prefecture,
    date_id,
    shop_id,
    event_type,
    event_json,
    event_hash,
    source_run_id,
    source_offset,
    source_response_hash,
    source_fetched_at,
    first_seen_at,
    last_changed_at,
    created_at,
    updated_at
)
VALUES (
    source.tournament_id,
    source.event_date,
    source.event_date_text,
    source.event_date_week,
    source.event_started_at,
    source.event_ended_at,
    source.tournament_name,
    source.venue_name,
    source.prefecture,
    source.date_id,
    source.shop_id,
    source.event_type,
    source.event_json,
    source.event_hash,
    source.source_run_id,
    source.source_offset,
    source.source_response_hash,
    source.source_fetched_at,
    source.source_fetched_at,
    source.source_fetched_at,
    current_timestamp(),
    current_timestamp()
)
""")

print(
    "Silver tournaments MERGE completed."
)

In [0]:
silver_tournament_count = (
    spark.table(
        TOURNAMENTS_TABLE
    )
    .filter(
        F.col("event_date")
        >= F.lit(
            MIN_EVENT_DATE
        ).cast("date")
    )
    .select(
        "tournament_id"
    )
    .distinct()
    .count()
)

print(
    "Silver tournament count:",
    silver_tournament_count,
)

In [0]:
missing_name_df = (
    spark.table(
        TOURNAMENTS_TABLE
    )
    .filter(
        F.col("event_date")
        >= F.lit(
            MIN_EVENT_DATE
        ).cast("date")
    )
    .filter(
        F.col(
            "tournament_name"
        ).isNull()
        | (
            F.trim(
                F.col("tournament_name")
            ) == ""
        )
    )
    .select(
        "tournament_id",
        "event_date",
        "tournament_name",
        "event_json",
    )
)

missing_name_count = (
    missing_name_df.count()
)

print(
    "Missing tournament names:",
    missing_name_count,
)

if missing_name_count > 0:
    display(
        missing_name_df.limit(10)
    )

In [0]:
display(
    spark.table(
        TOURNAMENTS_TABLE
    )
    .filter(
        F.col("event_date")
        >= F.lit(
            MIN_EVENT_DATE
        ).cast("date")
    )
    .select(
        "tournament_id",
        "event_date",
        "tournament_name",
        "event_type",
        "prefecture",
        "venue_name",
        "event_hash",
        "source_fetched_at",
        "updated_at",
    )
    .orderBy(
        F.col("event_date").desc(),
        "tournament_id",
    )
)